# Masked-Augmentation Mitigation — Colab GPU runner

Fine-tunes the CheXpert-pretrained DenseNet-121 with the shortcut (border) region masked, plus a no-masking **control**, then re-runs the AUROC + out-of-lung-localization (OLL) screens on pretrained / control / masked.

**Before running:** `Runtime → Change runtime type → GPU`.

**You need the data zip** (`chexpert_data.zip`, shared on Google Drive) — the images are gitignored, so they are not in the repo. Edit `DATA_ZIP` in cell 3 to point at where it lives in your Drive.

**What to send back:** the 6 CSVs written to `results/` (`mitigation_{pretrained,control,masked}_{auroc,oll}.csv`). Cell 7 zips them for download.

**Win condition:** the *masked* model shows lower OLL than both pretrained and control while its AUROC holds. A flat/null result is fine too — we report it honestly and the paper still ships on the diagnosis.

### 1. Clone the repo + install deps

In [ ]:
!git clone -b feat/mitigation https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib

### 2. Confirm the GPU is on

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU and re-run.'
print('GPU:', torch.cuda.get_device_name(0))

### 3. Mount Drive + unzip the data
Edit `DATA_ZIP` to the path of `chexpert_data.zip` in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to where you put the data zip in Drive >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/
!ls data/chexpert   # expect: PNG_train  PNG_valid  metadata_train.csv  metadata.csv

### 4. Train the two models (~minutes each on GPU)
`control` = plain fine-tune (no masking); `masked` = border masked on 50% of images.

In [ ]:
!python -m src.finetune --out checkpoints/control.pt --mask-frac 0.0 --epochs 4 --batch-size 16 --device cuda
!python -m src.finetune --out checkpoints/masked.pt  --mask-frac 0.5 --mask-kind border --epochs 4 --batch-size 16 --device cuda

### 5. Evaluate all three on AUROC + OLL

In [ ]:
!python -m src.mitigation_eval --tag pretrained                                  --device cuda
!python -m src.mitigation_eval --tag control --checkpoint checkpoints/control.pt --device cuda
!python -m src.mitigation_eval --tag masked  --checkpoint checkpoints/masked.pt  --device cuda

### 6. Quick look — the three-way comparison

In [ ]:
import pandas as pd, glob
for tag in ['pretrained', 'control', 'masked']:
    for kind in ['auroc', 'oll']:
        f = f'results/mitigation_{tag}_{kind}.csv'
        print(f'\n=== {tag} / {kind} ===')
        try:
            print(pd.read_csv(f).to_string(index=False))
        except FileNotFoundError:
            print('(missing)')

### 7. Zip the 6 CSVs to download and send to Jonathan

In [ ]:
!cd results && zip -q /content/mitigation_results.zip mitigation_*_auroc.csv mitigation_*_oll.csv && echo zipped
from google.colab import files
files.download('/content/mitigation_results.zip')